# Data Cleaning — UCI Drug Consumption Dataset

This notebook loads the raw UCI Drug Consumption dataset, assigns column names from the UCI documentation, inspects data quality, removes the fictitious drug Semeron (used to identify over-claimers), drops the ID column, and saves a clean version ready for EDA and encoding.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# The raw .data file has no header row.
# Column names come from the UCI documentation:
# https://archive.ics.uci.edu/dataset/373/drug+consumption+quantified

ALL_COLS = [
    'ID', 'Age', 'Gender', 'Education', 'Country', 'Ethnicity',
    'Nscore', 'Escore', 'Oscore', 'Ascore', 'Cscore', 'Impulsive', 'SS',
    'Alcohol', 'Amphet', 'Amyl', 'Benzos', 'Caff', 'Cannabis', 'Choc',
    'Coke', 'Crack', 'Ecstasy', 'Heroin', 'Ketamine', 'Legalh', 'LSD',
    'Meth', 'Mushrooms', 'Nicotine', 'Semer', 'VSA'
]

df = pd.read_csv('/Users/tg197682/Downloads/Computer Science Year 3/COMP3931_Individual_Project/student-addiction-risk-prediction/data/raw/drug_consumption.data', header=None, names=ALL_COLS)

print('Shape:', df.shape)
df.head()

Shape: (1885, 32)


,ID,Age,Gender,Education,Country,Ethnicity,Nscore,Escore,Oscore,Ascore,...,Ecstasy,Heroin,Ketamine,Legalh,LSD,Meth,Mushrooms,Nicotine,Semer,VSA
0,1,0.49788,0.48246,-0.05921,0.96082,0.12600,0.31287,-0.57545,-0.58331,-0.91699,...,CL0,CL0,CL0,CL0,CL0,CL0,CL0,CL2,CL0,CL0
1,2,-0.07854,-0.48246,1.98437,0.96082,-0.31685,-0.67825,1.93886,1.43533,0.76096,...,CL4,CL0,CL2,CL0,CL2,CL3,CL0,CL4,CL0,CL0
2,3,0.49788,-0.48246,-0.05921,0.96082,-0.31685,-0.46725,0.80523,-0.84732,-1.62090,...,CL0,CL0,CL0,CL0,CL0,CL0,CL1,CL0,CL0,CL0
3,4,-0.95197,0.48246,1.16365,0.96082,-0.31685,-0.14882,-0.80615,-0.01928,0.59042,...,CL0,CL0,CL2,CL0,CL0,CL0,CL0,CL2,CL0,CL0
4,5,0.49788,0.48246,1.98437,0.96082,-0.31685,0.73545,-1.63340,-0.45174,-0.30172,...,CL1,CL0,CL0,CL1,CL0,CL0,CL2,CL2,CL0,CL0


In [5]:
# Inspect data types
df.dtypes

ID             int64
Age          float64
Gender       float64
Education    float64
Country      float64
Ethnicity    float64
Nscore       float64
Escore       float64
Oscore       float64
Ascore       float64
Cscore       float64
Impulsive    float64
SS           float64
Alcohol          str
Amphet           str
Amyl             str
Benzos           str
Caff             str
Cannabis         str
Choc             str
Coke             str
Crack            str
Ecstasy          str
Heroin           str
Ketamine         str
Legalh           str
LSD              str
Meth             str
Mushrooms        str
Nicotine         str
Semer            str
VSA              str
dtype: object

In [ ]:
# Checking for missing values in the dataset
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Total missing:', df.isnull().sum().sum())

Missing values per column:
ID           0
Age          0
Gender       0
Education    0
Country      0
Ethnicity    0
Nscore       0
Escore       0
Oscore       0
Ascore       0
Cscore       0
Impulsive    0
SS           0
Alcohol      0
Amphet       0
Amyl         0
Benzos       0
Caff         0
Cannabis     0
Choc         0
Coke         0
Crack        0
Ecstasy      0
Heroin       0
Ketamine     0
Legalh       0
LSD          0
Meth         0
Mushrooms    0
Nicotine     0
Semer        0
VSA          0
dtype: int64

Total missing: 0


- The UCI Drug Consumption dataset contains **no missing values** — all 1,885 records are complete.
- The 12 demographic/personality features (Age through SS) are pre-quantified real values from the original study.
- The 19 drug columns use ordinal class labels: CL0–CL6.

In [7]:
# Describe numeric feature columns
FEATURE_COLS = ['Age','Gender','Education','Country','Ethnicity',
                'Nscore','Escore','Oscore','Ascore','Cscore','Impulsive','SS']

df[FEATURE_COLS].describe().round(3)

,Age,Gender,Education,Country,Ethnicity,Nscore,Escore,Oscore,Ascore,Cscore,Impulsive,SS
count,1885.000,1885.000,1885.000,1885.000,1885.000,1885.000,1885.000,1885.000,1885.000,1885.000,1885.000,1885.000
mean,0.035,-0.000,-0.004,0.356,-0.310,0.000,-0.000,-0.001,-0.000,-0.000,0.007,-0.003
std,0.878,0.483,0.950,0.700,0.166,0.998,0.997,0.996,0.997,0.998,0.954,0.964
min,-0.952,-0.482,-2.436,-0.570,-1.107,-3.464,-3.274,-3.274,-3.464,-3.464,-2.555,-2.078
25%,-0.952,-0.482,-0.611,-0.570,-0.317,-0.678,-0.695,-0.717,-0.606,-0.653,-0.711,-0.526
50%,-0.079,-0.482,-0.059,0.961,-0.317,0.043,0.003,-0.019,-0.017,-0.007,-0.217,0.080
75%,0.498,0.482,0.455,0.961,-0.317,0.630,0.638,0.723,0.761,0.585,0.530,0.765
max,2.592,0.482,1.984,0.961,1.907,3.274,3.274,2.902,3.464,3.464,2.902,1.922


In [8]:
# Drug usage class label distribution for each substance
DRUG_COLS = ['Alcohol','Amphet','Amyl','Benzos','Caff','Cannabis','Choc',
             'Coke','Crack','Ecstasy','Heroin','Ketamine','Legalh','LSD',
             'Meth','Mushrooms','Nicotine','Semer','VSA']

for col in DRUG_COLS:
    print(f'{col:12s}: {dict(df[col].value_counts().sort_index())}')

Alcohol     : {'CL0': np.int64(34), 'CL1': np.int64(34), 'CL2': np.int64(68), 'CL3': np.int64(198), 'CL4': np.int64(287), 'CL5': np.int64(759), 'CL6': np.int64(505)}
Amphet      : {'CL0': np.int64(976), 'CL1': np.int64(230), 'CL2': np.int64(243), 'CL3': np.int64(198), 'CL4': np.int64(75), 'CL5': np.int64(61), 'CL6': np.int64(102)}
Amyl        : {'CL0': np.int64(1305), 'CL1': np.int64(210), 'CL2': np.int64(237), 'CL3': np.int64(92), 'CL4': np.int64(24), 'CL5': np.int64(14), 'CL6': np.int64(3)}
Benzos      : {'CL0': np.int64(1000), 'CL1': np.int64(116), 'CL2': np.int64(234), 'CL3': np.int64(236), 'CL4': np.int64(120), 'CL5': np.int64(84), 'CL6': np.int64(95)}
Caff        : {'CL0': np.int64(27), 'CL1': np.int64(10), 'CL2': np.int64(24), 'CL3': np.int64(60), 'CL4': np.int64(106), 'CL5': np.int64(273), 'CL6': np.int64(1385)}
Cannabis    : {'CL0': np.int64(413), 'CL1': np.int64(207), 'CL2': np.int64(266), 'CL3': np.int64(211), 'CL4': np.int64(140), 'CL5': np.int64(185), 'CL6': np.int64(463)}

In [ ]:
# Removing Semeron (Semer) — a fictitious drug used to identify over-claimers.
# Respondents who claimed to have used Semeron are unreliable.
# UCI recommendation: flag and remove over-claimers (Semer != CL0).

overclaimer_mask = df['Semer'] != 'CL0'
print(f'Over-claimers identified (Semer != CL0): {overclaimer_mask.sum()}')

df_clean = df[~overclaimer_mask].copy()
df_clean = df_clean.drop(columns=['Semer'])  # drop Semer column — no longer needed
print(f'Rows after removing over-claimers: {len(df_clean)}')

Over-claimers identified (Semer != CL0): 8
Rows after removing over-claimers: 1877


In [ ]:
# Dropping the ID column — it is a record index with no predictive value
df_clean = df_clean.drop(columns=['ID'])
print('Columns after cleaning:')
print(df_clean.columns.tolist())
print('Shape:', df_clean.shape)

Columns after cleaning:
['Age', 'Gender', 'Education', 'Country', 'Ethnicity', 'Nscore', 'Escore', 'Oscore', 'Ascore', 'Cscore', 'Impulsive', 'SS', 'Alcohol', 'Amphet', 'Amyl', 'Benzos', 'Caff', 'Cannabis', 'Choc', 'Coke', 'Crack', 'Ecstasy', 'Heroin', 'Ketamine', 'Legalh', 'LSD', 'Meth', 'Mushrooms', 'Nicotine', 'VSA']
Shape: (1877, 30)


In [ ]:
# Checking for duplicate rows
dupes = df_clean.duplicated().sum()
print(f'Duplicate rows: {dupes}')

Duplicate rows: 0


- **8 over-claimers** removed (respondents who claimed to have used the fictitious drug Semeron).
- **ID column** dropped — purely an index with no predictive meaning.
- No duplicate rows found.
- Final clean dataset: **1,877 rows × 30 columns**.

In [12]:
# Save cleaned dataset
df_clean.to_csv('uci_drug_clean.csv', index=False)
print('Saved: uci_drug_clean.csv')
df_clean.shape

Saved: uci_drug_clean.csv


(1877, 30)